# 03 — Unified Transformation Layer (Silver → Silver)

**Position in pipeline:**
```
02_cleaning_and_validation_layer  (Bronze → Silver parquets)
        ↓
03_unified_transformation_layer   ← THIS NOTEBOOK
        ↓
Combined_Aggregation              (Silver tables → Gold)
```

**Inputs** — silver parquets written by `02_cleaning_and_validation_layer`:
| File | Grain |
|---|---|
| `crime_clean.parquet` | `(crime_id, lsoa_code)` / ASB rows |
| `claimant_clean.parquet` | `(lsoa_code, year)` |
| `adi_crime_clean.parquet` | `(lsoa_code, year)` |
| `health_clean.parquet` | `(lsoa_code, year)` |
| `houseprices_clean.parquet` | Transaction-level |
| `postcode_clean.parquet` | Postcode lookup |

**Outputs** — Delta tables in `crime_data.silver.*`:
| Spark Table | Description |
|---|---|
| `silver_<force_name>_crime` | One table per force detected in `crime_clean.parquet` |
| `silver_adi_components` | ADI: merged claimant + crime + health rates per LSOA/year |
| `silver_adi` | ADI: components + computed ADI score + engineered features |
| `silver_houseprice` | House prices aggregated to `(lsoa_code, year, month_num)` |
| `silver_postcode` | Postcode → LSOA lookup |

**Changes from previous version:**
- `quarter` and `quarter_label` columns removed as agreed by team
- `month` renamed to `month_num` for consistency across pipeline
- `force_name` used consistently (not `falls_within`)
- Crime tables are fully dynamic — no force names hardcoded


## 1. Imports & Configuration

In [0]:
import os
import numpy as np
import pandas as pd
from functools import reduce
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.window import Window

# ── Catalog / schema ──────────────────────────────────────────────────────────
CATALOG    = "crime_data"
SILVER_OUT = "silver"

# Silver Volume path — must match 02_cleaning_and_validation_layer
# FIXED: now uses Volume path consistent with cleaning layer
SILVER_DIR = f"/Volumes/{CATALOG}/silver/outputs"

# Input parquet paths (outputs from cleaning layer)
CRIME_IN     = f"{SILVER_DIR}/crime_clean.parquet"
CLAIMANT_IN  = f"{SILVER_DIR}/claimant_clean.parquet"
ADI_CRIME_IN = f"{SILVER_DIR}/adi_crime_clean.parquet"
HEALTH_IN    = f"{SILVER_DIR}/health_clean.parquet"
HP_IN        = f"{SILVER_DIR}/houseprices_clean.parquet"
PC_IN        = f"{SILVER_DIR}/postcode_clean.parquet"

# Enrichment output table names (fixed — not force-dependent)
OUT_ADI_COMP = f"{CATALOG}.{SILVER_OUT}.silver_adi_components"
OUT_ADI      = f"{CATALOG}.{SILVER_OUT}.silver_adi"
OUT_HP       = f"{CATALOG}.{SILVER_OUT}.silver_houseprice"
OUT_PC       = f"{CATALOG}.{SILVER_OUT}.silver_postcode"

# Crime output table names are built dynamically at runtime:
#   f"{CATALOG}.{SILVER_OUT}.silver_{force_name}_crime"
# Adding a new force CSV requires NO changes to this notebook.

# Home Office crime severity mapping (higher = more severe)
SEVERITY_MAP = {
    "Violence and sexual offences": 5,
    "Robbery":                       5,
    "Possession of weapons":         4,
    "Burglary":                      4,
    "Vehicle crime":                 3,
    "Criminal damage and arson":     3,
    "Drugs":                         3,
    "Theft from the person":         2,
    "Other theft":                   2,
    "Bicycle theft":                 2,
    "Shoplifting":                   2,
    "Public order":                  2,
    "Other crime":                   1,
    "Anti-social behaviour":         1,
}

print("✅ Configuration loaded")
print(f"   SILVER_DIR : {SILVER_DIR}")
print(f"   Catalog    : {CATALOG}.{SILVER_OUT}")


## 2. Shared Utilities

In [0]:
def check_row_counts(label, before, after):
    delta  = after - before
    pct    = round(after / before * 100, 1) if before else 0.0
    symbol = "✅" if after >= before * 0.95 else "⚠️"
    print(f"{symbol} {label}: {before:,} → {after:,}  (Δ {delta:+,}, kept {pct}%)")


def check_duplicates(df, keys, name):
    dup    = df.duplicated(subset=keys).sum()
    status = "✅ PASS" if dup == 0 else f"❌ FAIL — {dup:,} duplicates"
    print(f"  Duplicate check [{name}] on {keys}: {status}")
    assert dup == 0, f"[{name}] {dup:,} duplicates on {keys}"


def save_delta(pdf, table_name):
    """Convert a pandas DataFrame to Spark and write as a Delta table (overwrite)."""
    sdf = spark.createDataFrame(pdf)
    sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    saved = spark.table(table_name)
    print(f"  ✅ Saved → {table_name}  ({saved.count():,} rows, {len(saved.columns)} cols)")


print("✅ Utilities defined")


## 3. Crime Data — Feature Engineering & Dynamic Split by Force

**Inputs:** `crime_clean.parquet`

**Feature engineering (applied equally to every force):**
- `severity_tier` — integer 1–5 from Home Office crime taxonomy
- `is_asb` — boolean, True for Anti-social behaviour rows
- `has_location` — boolean, carried from cleaning layer

**Note:** `quarter` and `quarter_label` columns removed as agreed by team.

**Split & save:**
Forces are detected dynamically from the `force_name` column — no hardcoding.
Each force writes to `crime_data.silver.silver_<force_name>_crime`.


In [0]:
# ── Load crime_clean.parquet ─────────────────────────────────────────────────
crime_pdf = pd.read_parquet(CRIME_IN)
print(f"Loaded crime_clean.parquet: {len(crime_pdf):,} rows, {len(crime_pdf.columns)} columns")

# Detect forces present — dynamic, whatever was ingested
detected_forces = sorted(crime_pdf["force_name"].dropna().unique().tolist())
print(f"\nForces detected: {detected_forces}")
print(f"\nRow counts per force:")
print(crime_pdf["force_name"].value_counts().to_string())


In [0]:
# ── Feature engineering — applied once to all forces ─────────────────────────
def engineer_crime_features(df):
    df = df.copy()

    # 1. Severity tier (1 = low, 5 = high)
    df["severity_tier"] = df["crime_type"].map(SEVERITY_MAP).fillna(1).astype("int8")

    # 2. ASB flag
    df["is_asb"] = (df["crime_type"] == "Anti-social behaviour")

    # 3. Ensure has_location exists
    if "has_location" not in df.columns:
        df["has_location"] = df["longitude"].notna() & df["latitude"].notna()

    # 4. Normalise crime_type casing (defensive trim)
    df["crime_type"] = df["crime_type"].str.strip()

    # 5. Drop quarter columns as agreed by team
    df = df.drop(
        columns=["quarter", "quarter_label"],
        errors="ignore"
    )

    # 6. Ensure month column is named month_num for pipeline consistency
    if "month" in df.columns and "month_num" not in df.columns:
        df = df.rename(columns={"month": "month_num"})

    return df

crime_fe = engineer_crime_features(crime_pdf)

print(f"Feature engineering complete: {len(crime_fe):,} rows")
print(f"\nColumns: {list(crime_fe.columns)}")
print(f"\nSeverity tier distribution:")
print(crime_fe["severity_tier"].value_counts().sort_index().to_string())
print(f"\nASB rows: {crime_fe['is_asb'].sum():,}")


In [0]:
# ── Validation before split ───────────────────────────────────────────────────
print("=== NULL CHECK on key columns ===")
key_cols = [
    "force_name", "year", "month_num",
    "crime_type", "lsoa_code", "lsoa_name", "severity_tier"
]
for col in key_cols:
    if col in crime_fe.columns:
        n = crime_fe[col].isna().sum()
        print(f"  {'✅' if n == 0 else '⚠️'} {col}: {n:,} nulls")

print("\n=== YEAR RANGE ===")
print(crime_fe.groupby("year").size().sort_index().to_string())

print("\n=== CONFIRM QUARTER REMOVED ===")
quarter_present = "quarter" in crime_fe.columns
print(f"  {'⚠️ quarter still present' if quarter_present else '✅ quarter column removed'}")


In [0]:
# ── Column schema for all force silver tables ─────────────────────────────────
# FIXED: quarter and quarter_label removed, month_num used consistently
CRIME_SILVER_COLS = [
    "crime_id", "force_name",
    "year", "month_num", "month_name",
    "longitude", "latitude", "has_location", "location",
    "lsoa_code", "lsoa_name",
    "crime_type", "severity_tier", "is_asb",
    "last_outcome_category",
]

def select_available(df, cols):
    """Keep only columns that exist — handles schema differences between forces."""
    return df[[c for c in cols if c in df.columns]].copy()


# ── Dynamic split and save — one table per force ──────────────────────────────
crime_tables_written = {}

for force in detected_forces:
    force_df = select_available(
        crime_fe[crime_fe["force_name"] == force],
        CRIME_SILVER_COLS
    )

    assert len(force_df) > 0, f"❌ Force '{force}' produced empty DataFrame after split"

    table_name = f"{CATALOG}.{SILVER_OUT}.silver_{force}_crime"
    print(f"\nProcessing: {force}  ({len(force_df):,} rows)")
    save_delta(force_df, table_name)
    crime_tables_written[force] = table_name

print(f"\n✅ Crime silver tables written: {len(crime_tables_written)}")
for force, table in crime_tables_written.items():
    print(f"   {force:<30} → {table}")


## 4. ADI (Area Deprivation Index) — Feature Engineering

**Inputs:** `claimant_clean.parquet`, `adi_crime_clean.parquet`, `health_clean.parquet`

**Two output tables:**
1. `silver_adi_components` — merged component rates per `(lsoa_code, year)` — audit / analysis table
2. `silver_adi` — components + ADI score + normalised scores + deprivation band + YoY delta


In [0]:
# ── Load ADI component parquets ──────────────────────────────────────────────
claimant  = pd.read_parquet(CLAIMANT_IN)
adi_crime = pd.read_parquet(ADI_CRIME_IN)
health    = pd.read_parquet(HEALTH_IN)

print(f"claimant  : {len(claimant):,} rows   cols: {list(claimant.columns)}")
print(f"adi_crime : {len(adi_crime):,} rows   cols: {list(adi_crime.columns)}")
print(f"health    : {len(health):,} rows   cols: {list(health.columns)}")


In [0]:
# ── Inner-join all three components at (lsoa_code, year) ─────────────────────
claimant_base = len(claimant)

adi_components = (
    claimant
    .merge(adi_crime, on=["lsoa_code", "year"], how="inner")
    .merge(health,    on=["lsoa_code", "year"], how="inner")
)

check_row_counts("ADI inner-merge vs claimant baseline", claimant_base, len(adi_components))

adi_components = adi_components[[
    "lsoa_code", "lsoa_name", "pop", "year",
    "claimant_rate", "total_crime_rate", "total_prevalence_rate",
]].copy()

check_duplicates(adi_components, ["lsoa_code", "year"], "adi_components")

print(f"\nYear distribution:")
print(adi_components["year"].value_counts().sort_index().to_string())


In [0]:
# ── Save Table 1: silver_adi_components ──────────────────────────────────────
save_delta(adi_components, OUT_ADI_COMP)
print("  (component rates only — for audit and analysis)")


In [0]:
# ── Feature engineering for silver_adi (Table 2) ─────────────────────────────
adi = adi_components.copy()

# 1. Composite ADI score
adi["ADI"] = (
    adi["claimant_rate"] + adi["total_crime_rate"] + adi["total_prevalence_rate"]
).round(4)

# 2. Min-max normalise each component within year (0-1 scale per year)
for rate_col, norm_col in [
    ("claimant_rate",         "claimant_rate_norm"),
    ("total_crime_rate",      "crime_rate_norm"),
    ("total_prevalence_rate", "health_rate_norm"),
]:
    mn = adi.groupby("year")[rate_col].transform("min")
    mx = adi.groupby("year")[rate_col].transform("max")
    adi[norm_col] = ((adi[rate_col] - mn) / (mx - mn).replace(0, np.nan)).round(4)

# 3. ADI normalised within year
mn = adi.groupby("year")["ADI"].transform("min")
mx = adi.groupby("year")["ADI"].transform("max")
adi["ADI_norm"] = ((adi["ADI"] - mn) / (mx - mn).replace(0, np.nan)).round(4)

# 4. Deprivation band — quintile within each year (1=least, 5=most deprived)
adi["deprivation_band"] = (
    adi.groupby("year")["ADI"]
       .transform(lambda x: pd.qcut(x, q=5, labels=[1,2,3,4,5], duplicates="drop"))
       .astype("Int8")
)

# 5. Year-over-year ADI delta per LSOA
adi = adi.sort_values(["lsoa_code", "year"])
adi["adi_yoy_delta"] = adi.groupby("lsoa_code")["ADI"].diff().round(4)

print(f"ADI feature engineering complete: {len(adi):,} rows")
print(f"Columns: {list(adi.columns)}")
print(f"\nADI score summary:")
print(adi["ADI"].describe().round(4).to_string())
print(f"\nDeprivation band distribution:")
print(adi["deprivation_band"].value_counts().sort_index().to_string())

n_yoy_null = adi["adi_yoy_delta"].isna().sum()
print(f"\nℹ️  adi_yoy_delta nulls: {n_yoy_null:,} (expected — first year per LSOA has no prior year)")


In [0]:
# ── Save Table 2: silver_adi ──────────────────────────────────────────────────
save_delta(adi, OUT_ADI)
print("  (full feature set including ADI score — downstream join table)")
print()
print(f"  silver_adi_components : {len(adi_components):,} rows — component rates only")
print(f"  silver_adi            : {len(adi):,} rows — components + ADI score + features")


## 5. House Prices — Feature Engineering & Transformation

**Inputs:** `houseprices_clean.parquet`, `postcode_clean.parquet`

**Transformations:**
- Left-join transactions to LSOA via postcode lookup
- Derive `year` and `month_num` from `date_of_transfer`
- Drop rows with no LSOA match
- Aggregate to `(lsoa_code, year, month_num)`: `median_price`, `mean_price`, `transaction_count`
- `log_median_price` — log-transformed for regression-ready output
- `price_band` — quintile within year (1=cheapest, 5=most expensive)

**Output table:** `silver_houseprice`

**Note:** month column renamed to `month_num` for pipeline consistency.


In [0]:
hp        = pd.read_parquet(HP_IN)
postcodes = pd.read_parquet(PC_IN)

hp["date_of_transfer"] = pd.to_datetime(hp["date_of_transfer"], errors="raise")
hp["postcode"]         = hp["postcode"].astype("string")
hp["price"]            = hp["price"].astype("int64")
postcodes["postcode"]  = postcodes["postcode"].astype("string")
postcodes["lsoa_code"] = postcodes["lsoa_code"].astype("string")

print(f"houseprices_clean : {len(hp):,} rows   cols: {list(hp.columns)}")
print(f"postcode_clean    : {len(postcodes):,} rows   cols: {list(postcodes.columns)}")


In [0]:
# Validate postcode lookup is 1:1 then join
assert not postcodes["postcode"].duplicated().any(), "postcode_clean has duplicate postcodes"

before_join = len(hp)
hp = hp.merge(postcodes, on="postcode", how="left")
check_row_counts("HP left-join postcode lookup", before_join, len(hp))
assert len(hp) == before_join, "❌ Fan-out detected — postcode lookup is not 1:1"

unmatched = hp["lsoa_code"].isna().sum()
print(f"ℹ️  Unmatched postcodes: {unmatched:,} ({unmatched / len(hp):.2%}) — will be dropped")


In [0]:
# Derive year/month_num and drop unassignable rows
# FIXED: column named month_num for pipeline consistency
hp["year"]      = hp["date_of_transfer"].dt.year.astype("int16")
hp["month_num"] = hp["date_of_transfer"].dt.month.astype("int8")

before_drop = len(hp)
hp = hp.dropna(subset=["lsoa_code"]).reset_index(drop=True)
check_row_counts("Drop unmatched LSOA rows", before_drop, len(hp))


In [0]:
# Aggregate to (lsoa_code, year, month_num)
# FIXED: group by month_num not month
hp_agg = (
    hp.groupby(["lsoa_code", "year", "month_num"], as_index=False)
      .agg(
          median_price      =("price", "median"),
          mean_price        =("price", "mean"),
          transaction_count =("price", "size"),
      )
)
hp_agg["median_price"]      = hp_agg["median_price"].round(0).astype("int64")
hp_agg["mean_price"]        = hp_agg["mean_price"].round(0).astype("int64")
hp_agg["transaction_count"] = hp_agg["transaction_count"].astype("int64")

check_row_counts("HP aggregation (transaction → LSOA/year/month_num)", len(hp), len(hp_agg))

# Feature engineering
hp_agg["log_median_price"] = np.log(hp_agg["median_price"].replace(0, np.nan)).round(4)
hp_agg["price_band"] = (
    hp_agg.groupby("year")["median_price"]
          .transform(lambda x: pd.qcut(x, q=5, labels=[1,2,3,4,5], duplicates="drop"))
          .astype("Int8")
)

check_duplicates(hp_agg, ["lsoa_code", "year", "month_num"], "hp_agg")
assert (hp_agg["transaction_count"] >= 1).all(), "transaction_count must be >= 1"
print(f"✅ House prices feature engineering complete: {len(hp_agg):,} rows")
print(f"   Columns: {list(hp_agg.columns)}")


In [0]:
save_delta(hp_agg, OUT_HP)
print("✅ silver_houseprice saved")


## 6. Postcode Lookup — Enrichment & Save

**Input:** `postcode_clean.parquet`
Adds `postcode_area` (e.g. `CB`) and `postcode_district` (e.g. `CB1`) for spatial grouping.
**Output table:** `silver_postcode`


In [0]:
pc = pd.read_parquet(PC_IN).copy()

pc["postcode_area"]     = pc["postcode"].str.extract(r"^([A-Z]{1,2})")
pc["postcode_district"] = pc["postcode"].str.extract(r"^([A-Z]{1,2}[0-9]{1,2})")

assert not pc["postcode"].duplicated().any(), "❌ Duplicate postcodes after enrichment"
print(f"✅ Postcode uniqueness confirmed — {len(pc):,} rows")
print(f"\nTop 10 postcode areas:")
print(pc["postcode_area"].value_counts().head(10).to_string())

save_delta(pc, OUT_PC)
print("✅ silver_postcode saved")


## 7. Run Summary & Final Validation

In [0]:
# ── Build the full output table registry dynamically ─────────────────────────
all_output_tables = {}

for force, table in crime_tables_written.items():
    all_output_tables[f"silver_{force}_crime"] = table

all_output_tables["silver_adi_components"] = OUT_ADI_COMP
all_output_tables["silver_adi"]            = OUT_ADI
all_output_tables["silver_houseprice"]     = OUT_HP
all_output_tables["silver_postcode"]       = OUT_PC

# ── Read back every table and report ─────────────────────────────────────────
print("=" * 70)
print("UNIFIED TRANSFORMATION LAYER — RUN SUMMARY")
print("=" * 70)

all_passed = True
for label, table in all_output_tables.items():
    try:
        df    = spark.table(table)
        rows  = df.count()
        cols  = len(df.columns)
        # Confirm quarter column is absent from crime tables
        quarter_check = ""
        if "crime" in label:
            quarter_check = "  ✅ no quarter" if "quarter" not in df.columns else "  ⚠️ quarter still present"
        print(f"  ✅ {label:<40} {rows:>10,} rows   {cols:>3} cols{quarter_check}")
    except Exception as e:
        print(f"  ❌ {label:<40} ERROR: {e}")
        all_passed = False

print("=" * 70)
if all_passed:
    print("\n✅ All tables written successfully — ready for Combined_Aggregation.")
else:
    print("\n❌ One or more tables failed — check errors above before running Combined_Aggregation.")

print("\nForce tables available for Combined_Aggregation:")
for force, table in crime_tables_written.items():
    print(f"  spark.table('{table}')")
